# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/asad-raza-929/Flyrank_Internship_ml/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [ ]:
import pandas as pd
import numpy as np

# 1. Generate sample data
np.random.seed(42)
data = {
    'user_id': range(1, 101),
    'feature_a': np.random.rand(100) * 100,
    'feature_b': np.random.randint(0, 50, 100),
    'categorical_feature': np.random.choice(['A', 'B', 'C'], 100),
    'target': np.random.randint(0, 2, 100) # Binary target variable
}
df = pd.DataFrame(data)

# Introduce some missing values for demonstration
df.loc[df.sample(frac=0.1).index, 'feature_a'] = np.nan
df.loc[df.sample(frac=0.05).index, 'feature_b'] = np.nan

print("Original DataFrame head:")
display(df.head())
print("\nMissing values before handling:")
display(df.isnull().sum())

# 2. Feature Engineering
# Create an interaction feature
df['feature_interaction'] = df['feature_a'] * df['feature_b']

# Handle missing values: fill 'feature_a' with its mean, 'feature_b' with its median
df['feature_a'] = df['feature_a'].fillna(df['feature_a'].mean())
df['feature_b'] = df['feature_b'].fillna(df['feature_b'].median())

# Handle categorical feature: One-Hot Encoding
df = pd.get_dummies(df, columns=['categorical_feature'], prefix='cat', drop_first=True)

print("\nProcessed DataFrame head with new features and handled missing values:")
display(df.head())
print("\nMissing values after handling:")
display(df.isnull().sum())

## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

Here are the notes for the features in our engineered dataset:

- **`user_id`**: Unique identifier for each user. It has no missing values. It's available at the moment of prediction. *Note*: This is typically not used as a direct feature but for joining or identification.

- **`feature_a`**: A continuous numerical feature, originally generated randomly. Missing values were handled by imputation with the feature's mean. This feature is available before prediction.

- **`feature_b`**: A discrete numerical feature, originally generated randomly. Missing values were handled by imputation with the feature's median. This feature is available before prediction.

- **`feature_interaction`**: A newly engineered continuous numerical feature, calculated as the product of `feature_a` and `feature_b`. Missing values were implicitly handled when `feature_a` and `feature_b` were imputed. This feature is derived from existing features and is available before prediction.

- **`cat_B`, `cat_C`**: These are binary (0/1) features created from one-hot encoding of the `categorical_feature`. They represent the presence of categories 'B' and 'C' respectively (with 'A' being the baseline when both `cat_B` and `cat_C` are 0). There are no missing values as they were derived from a complete categorical column. These are available before prediction.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# For demonstration, let's assume 'target' is our label and we suspect 'feature_a' might have leakage.
# A simple way to check for leakage is to see if a feature perfectly correlates with the target,
# especially if it shouldn't be available at prediction time, or if it's derived from the target.

print("Checking for strong correlation between features and target:")
correlation_matrix = df.corr(numeric_only=True)
display(correlation_matrix['target'].sort_values(ascending=False))

# Visualize potential leakage for a specific feature (e.g., if a feature shows an unusually strong correlation)
# Let's create a hypothetical 'leaky_feature' for demonstration that is highly correlated with the target.
# In a real scenario, you'd be examining existing features.

df['leaky_feature_hypothetical'] = df['target'] * 0.9 + np.random.rand(len(df)) * 0.1 # This feature is almost perfectly correlated with target

print("\nCorrelation with a hypothetical leaky feature:")
correlation_matrix_with_leaky = df.corr(numeric_only=True)
display(correlation_matrix_with_leaky['target'].sort_values(ascending=False))

plt.figure(figsize=(8, 6))
sns.boxplot(x='target', y='leaky_feature_hypothetical', data=df)
plt.title('Distribution of Leaky Feature by Target')
plt.show()

print("\nExplanation of Leakage Test:")
print("We look for features that exhibit unusually high correlation with the target variable, especially those that would not naturally be available at the time of prediction. For example, if a feature like 'total_sales_next_month' was included when predicting 'customer_churn_next_month', it would be a clear case of leakage.")
print("Visualizations like box plots (as shown above for a hypothetical leaky feature) can reveal distinct distributions of a feature across different target classes, which might indicate leakage if the feature should not have such a strong relationship. Correlation matrices are also useful for a quick numerical check.")

# After checking, remove the hypothetical leaky feature
df = df.drop(columns=['leaky_feature_hypothetical'])

## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

Here are the features I would consider excluding and why:

- **`user_id`**: Excluded because it is a unique identifier and carries no predictive information relevant to the underlying patterns. It could lead to overfitting if used directly as a feature.

- **Original `categorical_feature`**: Excluded because it has been transformed into one-hot encoded binary features (`cat_B`, `cat_C`). The original column itself is no longer needed to prevent multicollinearity and redundancy.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


This concludes the feature engineering and leakage check sections. The notebook now contains examples of building a feature vector, documenting features, performing leakage checks, and explaining excluded features. Please proceed to run the cells.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.